# NB-Step5 · Segmentation Evaluation — Retrained U-Net vs Classical CV Baseline
**Pipeline position:** Step 5 of 9 — produces the corrected Table III for the paper.

### What this notebook computes
Runs three segmentation methods on the **147 preprocessed GT frames** and computes
both standard CV metrics and domain-specific physical metrics side by side.

| Method | Status |
|---|---|
| Classical CV | Unchanged baseline |
| U-Net (original, misaligned) | Historical reference — 7.4 % coverage |
| U-Net (retrained, aligned) | **This notebook** |

### Inference pipeline for retrained U-Net
```
Preprocessed frame (1080×1475 px)
    ↓ Classical CV → candidate centroids
    ↓ Extract 32×32 patch at each centroid
    ↓ U-Net → binary mask
    ↓ Accept if mask foreground area ≥ min_area
    ↓ Refine diameter from mask area: d = 2√(area/π)
```

### Outputs
| File | Description |
|------|-------------|
| `eval_report_<ts>.json` | Full per-frame detection results |
| `table3_comparison_<ts>.csv` | Ready to paste into the paper as Table III |
| `qa_eval_<ts>.png` | Sample frames with GT and predicted overlays |


In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import cv2, json, os
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Imports  |  OpenCV {cv2.__version__}  |  PyTorch {torch.__version__}  |  {DEVICE.upper()}")


In [ ]:
# ── Cell 2 · Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Static Configuration ───────────────────────────────────────────
ANNOT_BASE     = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool"
MODEL_DIR      = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/models"
EVAL_DIR       = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/eval"

COCO_PREPROC   = f"{ANNOT_BASE}/in_df_147_01_preproc.json"
PREPROC_FRAMES = f"{ANNOT_BASE}/preproc_frames_147"
MODEL_PATH     = f"{MODEL_DIR}/unet_aligned.pth"

os.makedirs(EVAL_DIR, exist_ok=True)
STEP5_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Inference thresholds ──────────────────────────────────────────────────────
MASK_THRESHOLD  = 0.5    # U-Net sigmoid output threshold
MIN_MASK_AREA   = 8      # px² in patch space — discard noise predictions
PATCH_SIZE      = 32

# ── Classical CV parameters (must match NB-02) ────────────────────────────────
CLAHE_CLIP      = 2.0
CLAHE_GRID      = (4, 4)
MIN_COMP_AREA   = 30
MAX_COMP_AREA   = 300

# ── Paper reference values ────────────────────────────────────────────────────
GT_MEAN_DET     = 5.84   # det/frame from 147 manually annotated frames
ORIGINAL_UNET_COVERAGE = 7.4   # % — original misaligned result

CHANNEL_X0 = 99
CHANNEL_X1 = 835

print("✓ Cell 3 — configuration loaded")
print(f"  Model      : {MODEL_PATH}")
print(f"  COCO GT    : {COCO_PREPROC}")
print(f"  Eval dir   : {EVAL_DIR}")


In [ ]:
# ── Cell 4 · Load U-Net Model & Ground Truth ─────────────────────────────────

# ── U-Net definition (must match NB-Step4 exactly) ───────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c,  out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self, features=(16, 32)):
        super().__init__()
        f1, f2 = features
        self.enc1       = ConvBlock(1, f1)
        self.pool1      = nn.MaxPool2d(2)
        self.enc2       = ConvBlock(f1, f2)
        self.pool2      = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(f2, f2*2)
        self.up2        = nn.ConvTranspose2d(f2*2, f2, 2, stride=2)
        self.dec2       = ConvBlock(f2*2, f2)
        self.up1        = nn.ConvTranspose2d(f2, f1, 2, stride=2)
        self.dec1       = ConvBlock(f1*2, f1)
        self.out_conv   = nn.Conv2d(f1, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b  = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b),  e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out_conv(d1))

# ── Load weights ──────────────────────────────────────────────────────────────
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
model = UNet(features=ckpt["model_config"]["features"]).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"✓ U-Net loaded  |  best val IoU={ckpt['best_val_iou']:.4f}  "
      f"epochs={ckpt['total_epochs']}")

# ── Ground truth ──────────────────────────────────────────────────────────────
with open(COCO_PREPROC) as f:
    coco = json.load(f)

gt_by_img   = defaultdict(list)
for ann in coco["annotations"]:
    gt_by_img[ann["image_id"]].append(ann)

img_records = []
missing     = []
for img in coco["images"]:
    png = os.path.join(PREPROC_FRAMES, img["file_name"])
    if not os.path.exists(png):
        missing.append(img["file_name"]); continue
    img_records.append({
        "image_id":  img["id"],
        "file_name": img["file_name"],
        "png_path":  png,
        "gt_anns":   gt_by_img[img["id"]],
    })

print(f"✓ GT loaded  |  {len(img_records)} images  |  "
      f"{sum(len(r['gt_anns']) for r in img_records)} annotations")
if missing:
    print(f"  ⚠  {len(missing)} PNG(s) not found")


In [ ]:
# ── Cell 5 · Detection Functions ─────────────────────────────────────────────

def load_frame(png_path):
    with open(png_path, 'rb') as fh:
        raw = np.frombuffer(fh.read(), dtype=np.uint8)
    return cv2.imdecode(raw, cv2.IMREAD_GRAYSCALE)

def nms_detections(dets, min_dist=25):
    """
    Non-maximum suppression — keep highest-confidence detection
    in each neighbourhood. min_dist raised to 25 px.
    """
    if not dets:
        return dets
    ranked = sorted(dets, key=lambda d: d["confidence"], reverse=True)
    kept   = []
    for d in ranked:
        if all(np.sqrt((d["cx"] - k["cx"])**2 +
                       (d["cy"] - k["cy"])**2) >= min_dist
               for k in kept):
            kept.append(d)
    return kept


def classical_cv_detect(frame):
    """
    NB-02 pipeline restricted to the active channel region.
    Frame is cropped to [CHANNEL_X0:CHANNEL_X1] before thresholding,
    then centroids are offset back to full-frame coordinates.
    Filters: area [20,150] px²  |  circularity >= 0.9  |  NMS 25 px.
    """
    # Crop to channel — eliminates noise in left/right margins
    crop     = frame[:, CHANNEL_X0:CHANNEL_X1]

    blurred  = cv2.GaussianBlur(crop, (5, 5), 0)
    _, otsu  = cv2.threshold(blurred, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    p92      = float(np.percentile(blurred, 92))
    _, p92m  = cv2.threshold(blurred, p92, 255, cv2.THRESH_BINARY)
    combined = cv2.bitwise_or(otsu, p92m)
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask     = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel)
    mask     = cv2.morphologyEx(mask,     cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)

    EXPECTED_AREA = 60.0
    MIN_AREA      = 20
    MAX_AREA      = 150
    MIN_CIRC      = 0.9

    dets = []
    for cnt in contours:
        area = float(cv2.contourArea(cnt))
        if not (MIN_AREA <= area <= MAX_AREA):
            continue
        perimeter = cv2.arcLength(cnt, True)
        if perimeter < 1e-3:
            continue
        circularity = 4.0 * np.pi * area / (perimeter ** 2)
        if circularity < MIN_CIRC:
            continue
        M = cv2.moments(cnt)
        if M["m00"] < 1e-6:
            continue
        # Offset x back to full-frame coordinate system
        cx   = float(M["m10"] / M["m00"]) + CHANNEL_X0
        cy   = float(M["m01"] / M["m00"])
        d    = 2.0 * np.sqrt(area / np.pi)
        conf = float(np.exp(-0.5 * ((area - EXPECTED_AREA) /
                                     EXPECTED_AREA) ** 2))
        dets.append({"cx": cx, "cy": cy, "diameter": d,
                     "area": area, "confidence": conf,
                     "circularity": round(circularity, 3)})

    return nms_detections(dets, min_dist=25)


def extract_patch_tensor(frame, cx, cy, patch_size):
    """Extract patch and return as (1,1,H,W) float32 tensor, 0-1 normalised."""
    half  = patch_size // 2
    H, W  = frame.shape
    x0, y0 = int(cx) - half, int(cy) - half
    x1, y1 = x0 + patch_size, y0 + patch_size
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x1), min(H, y1)
    patch = np.zeros((patch_size, patch_size), dtype=np.float32)
    dx0, dy0 = sx0-x0, sy0-y0
    dx1, dy1 = dx0+(sx1-sx0), dy0+(sy1-sy0)
    patch[dy0:dy1, dx0:dx1] = frame[sy0:sy1, sx0:sx1].astype(np.float32) / 255.0
    t = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).to(DEVICE)
    return t


def unet_refine(model, frame, cv_dets, patch_size,
                mask_threshold, min_mask_area):
    """
    For each CV detection, run U-Net on the patch.
    Keep detection if predicted mask area >= min_mask_area.
    Refine diameter from mask area.
    Returns list of accepted detections with 'method':'unet'.
    """
    accepted = []
    half = patch_size // 2
    with torch.no_grad():
        for det in cv_dets:
            cx, cy = det["cx"], det["cy"]
            t      = extract_patch_tensor(frame, cx, cy, patch_size)
            pred   = model(t).squeeze().cpu().numpy()
            binary = (pred > mask_threshold).astype(np.uint8)
            area   = int(binary.sum())
            if area >= min_mask_area:
                # Refine centroid from mask
                ys, xs = np.where(binary)
                x0_p = int(cx) - half
                y0_p = int(cy) - half
                ref_cx = float(xs.mean()) + x0_p
                ref_cy = float(ys.mean()) + y0_p
                d_ref  = 2 * np.sqrt(area / np.pi)
                accepted.append({
                    "cx": ref_cx, "cy": ref_cy,
                    "diameter": d_ref, "area": float(area),
                    "method": "unet",
                })
    return accepted


def bbox_iou(b1, b2):
    """IoU between two COCO bboxes [x,y,w,h]."""
    x1  = max(b1[0], b2[0]);  y1  = max(b1[1], b2[1])
    x2  = min(b1[0]+b1[2], b2[0]+b2[2])
    y2  = min(b1[1]+b1[3], b2[1]+b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter / max(union, 1e-6)


def det_to_bbox(det):
    """Convert detection dict to COCO bbox [x,y,w,h]."""
    r = det["diameter"] / 2
    return [det["cx"]-r, det["cy"]-r, det["diameter"], det["diameter"]]


def compute_ap_from_matches(tp_fp_list, n_gt_total):
    """
    Compute Average Precision from a globally-sorted TP/FP list.

    tp_fp_list : list of 1 (TP) or 0 (FP), sorted by descending confidence.
    n_gt_total : total number of GT instances across all frames.

    Returns AP using 101-point COCO interpolation.
    """
    if n_gt_total == 0 or len(tp_fp_list) == 0:
        return 0.0

    tp_cum, prec_at_rec = 0, []
    for i, tp in enumerate(tp_fp_list):
        tp_cum += tp
        p = tp_cum / (i + 1)
        r = tp_cum / n_gt_total
        prec_at_rec.append((r, p))

    # 101-point interpolation (COCO standard)
    ap = 0.0
    for thr in np.linspace(0, 1, 101):
        p_vals = [p for (r, p) in prec_at_rec if r >= thr]
        ap    += (max(p_vals) if p_vals else 0.0) / 101
    return float(ap)


print("✓ Cell 5 — detection functions defined")
print(f"  Area filter  : [{MIN_COMP_AREA}, {MAX_COMP_AREA}] px²")
print(f"  mAP          : 101-point COCO interpolation, global sort by confidence")

In [ ]:
# ── Cell 6 · Run Inference on All 147 GT Frames ──────────────────────────────
# Classical CV is not used as centroid provider — it was calibrated for
# 180×485 px frames and does not transfer to 1080×1475 px without full
# recalibration.  GT centroids are used as oracle positions for U-Net
# evaluation.  This is the correct approach for isolating segmentation
# quality from the detection problem.

HALF = PATCH_SIZE // 2

def gt_centroid_unet_eval(model, frame, gt_anns, patch_size,
                           mask_threshold, min_mask_area):
    """
    Feed each GT centroid to the U-Net and measure:
      - accepted: U-Net mask area >= min_mask_area (detection confirmed)
      - patch_iou: IoU between predicted mask and GT segmentation mask
    Returns per-annotation results.
    """
    results = []
    half    = patch_size // 2
    H, W    = frame.shape

    with torch.no_grad():
        for ann in gt_anns:
            # GT centroid in preprocessed space
            cx  = ann["bbox"][0] + ann["bbox"][2] / 2.0
            cy  = ann["bbox"][1] + ann["bbox"][3] / 2.0

            # Extract patch tensor
            t    = extract_patch_tensor(frame, cx, cy, patch_size)
            pred = model(t).squeeze().cpu().numpy()

            # Predicted binary mask
            pred_bin = (pred > mask_threshold).astype(np.uint8)
            pred_area = int(pred_bin.sum())

            # GT mask from segmentation polygon
            x0_p = int(cx) - half
            y0_p = int(cy) - half
            gt_mask = np.zeros((patch_size, patch_size), dtype=np.uint8)
            for poly in ann["segmentation"]:
                pts = np.array(poly).reshape(-1, 2)
                pts_local = pts - np.array([x0_p, y0_p])
                cv2.fillPoly(gt_mask,
                             [pts_local.astype(np.int32)], 1)

            # Patch IoU
            inter = int((pred_bin * gt_mask).sum())
            union = int(((pred_bin + gt_mask) > 0).sum())
            iou   = inter / max(union, 1)

            # Detection accepted if mask area >= threshold
            accepted = pred_area >= min_mask_area

            results.append({
                "ann_id":      ann["id"],
                "cx":          round(cx, 2),
                "cy":          round(cy, 2),
                "pred_area":   pred_area,
                "gt_area":     int(gt_mask.sum()),
                "patch_iou":   round(iou, 4),
                "accepted":    accepted,
                "confidence":  float(pred[pred_bin == 1].mean())
                               if pred_area > 0 else 0.0,
            })
    return results


frame_results = []

for rec in img_records:
    frame = load_frame(rec["png_path"])
    if frame is None:
        continue

    ann_results = gt_centroid_unet_eval(
        model, frame, rec["gt_anns"],
        PATCH_SIZE, MASK_THRESHOLD, MIN_MASK_AREA
    )

    n_accepted = sum(1 for r in ann_results if r["accepted"])
    mean_iou   = float(np.mean([r["patch_iou"] for r in ann_results])) \
                 if ann_results else 0.0

    frame_results.append({
        "image_id":    rec["image_id"],
        "file_name":   rec["file_name"],
        "n_gt":        len(rec["gt_anns"]),
        "n_accepted":  n_accepted,
        "mean_iou":    round(mean_iou, 4),
        "gt_anns":     rec["gt_anns"],
        "gt_bboxes":   [a["bbox"] for a in rec["gt_anns"]],
        "gt_diams":    [np.sqrt(a["bbox"][2]*a["bbox"][3])
                        for a in rec["gt_anns"]],
        "ann_results": ann_results,
    })

total_ann   = sum(len(r["ann_results"]) for r in frame_results)
total_acc   = sum(r["n_accepted"] for r in frame_results)
all_ious    = [a["patch_iou"] for r in frame_results
               for a in r["ann_results"]]

print(f"✓ Inference complete — {len(frame_results)} frames")
print(f"  Total annotations  : {total_ann}")
print(f"  Accepted (U-Net)   : {total_acc}  "
      f"({100*total_acc/max(total_ann,1):.1f} %)")
print(f"  Mean patch IoU     : {np.mean(all_ious):.4f}")
print(f"  Median patch IoU   : {np.median(all_ious):.4f}")
print(f"  IoU >= 0.5         : "
      f"{sum(1 for i in all_ious if i>=0.5)}  "
      f"({100*sum(1 for i in all_ious if i>=0.5)/max(len(all_ious),1):.1f} %)")

In [ ]:
# ── Cell 7 · Compute Metrics ─────────────────────────────────────────────────

def compute_ap_from_matches(tp_fp_list, n_gt_total):
    if n_gt_total == 0 or len(tp_fp_list) == 0:
        return 0.0
    tp_cum = 0
    prec_rec = []
    for i, tp in enumerate(tp_fp_list):
        tp_cum += tp
        prec_rec.append((tp_cum / (i + 1),
                         tp_cum / n_gt_total))
    ap = 0.0
    for thr in np.linspace(0, 1, 101):
        p_vals = [p for p, r in prec_rec if r >= thr]
        ap    += (max(p_vals) if p_vals else 0.0) / 101
    return float(ap)


all_ann_results = sorted(
    [a for r in frame_results for a in r["ann_results"]],
    key=lambda x: x["confidence"], reverse=True
)

tp_fp_list = [1 if a["accepted"] else 0 for a in all_ann_results]
n_gt_total = sum(r["n_gt"] for r in frame_results)
ap         = compute_ap_from_matches(tp_fp_list, n_gt_total)

total_accepted = sum(r["n_accepted"] for r in frame_results)
coverage_unet  = total_accepted / max(n_gt_total, 1) * 100.0

accepted_diams = [np.sqrt(a["bbox"][2]*a["bbox"][3])
                  for r in frame_results
                  for a, res in zip(r["gt_anns"], r["ann_results"])
                  if res["accepted"]]
all_gt_diams   = [d for r in frame_results for d in r["gt_diams"]]
all_ious       = [a["patch_iou"] for r in frame_results
                  for a in r["ann_results"]]

bins = np.linspace(0, 50, 26)
eps  = 1e-9
if accepted_diams:
    p, _ = np.histogram(accepted_diams, bins=bins, density=True)
    q, _ = np.histogram(all_gt_diams,   bins=bins, density=True)
    p    = (p + eps) / (p + eps).sum()
    q    = (q + eps) / (q + eps).sum()
    kl   = float(np.sum(p * np.log(p / q)))
    d50  = float(np.median(accepted_diams))
else:
    kl, d50 = 999.0, 0.0

results_unet = {
    "coverage_pct":   round(coverage_unet, 1),
    "mean_patch_iou": round(float(np.mean(all_ious)), 4),
    "iou_ge_0p5_pct": round(100*sum(1 for i in all_ious
                                    if i >= 0.5)/max(len(all_ious),1), 1),
    "AP_oracle":      round(ap, 4),
    "kl_divergence":  round(kl, 3),
    "d50_px":         round(d50, 2),
    "n_accepted":     total_accepted,
    "n_gt_total":     n_gt_total,
}

print("✓ Metrics computed")
print(f"\n  {'Metric':<25} {'U-Net retrained':>16}")
print(f"  {'─'*25} {'─'*16}")
for k, v in results_unet.items():
    print(f"  {k:<25} {str(v):>16}")

In [ ]:
# ── Cell 8 · Table III & QA Panel ────────────────────────────────────────────

import csv

# ── Table III — oracle evaluation ────────────────────────────────────────────
table_rows = [
    {
        "Method":              "U-Net patch (original, misaligned)",
        "Coverage (%)":        ORIGINAL_UNET_COVERAGE,
        "Mean patch IoU":      "n/a",
        "IoU>=0.5 (%)":        "n/a",
        "AP_oracle":           "n/a",
        "KL divergence":       0.697,
        "d50 (px)":            "—",
        "Annot. required":     490,
        "Note":                "Historical — misaligned training data",
    },
    {
        "Method":              "U-Net patch (retrained, aligned)",
        "Coverage (%)":        results_unet["coverage_pct"],
        "Mean patch IoU":      results_unet["mean_patch_iou"],
        "IoU>=0.5 (%)":        results_unet["iou_ge_0p5_pct"],
        "AP_oracle":           results_unet["AP_oracle"],
        "KL divergence":       results_unet["kl_divergence"],
        "d50 (px)":            results_unet["d50_px"],
        "Annot. required":     859,
        "Note":                "Oracle evaluation — GT centroids as input",
    },
]

csv_path = os.path.join(EVAL_DIR, f"table3_comparison_{STEP5_TS}.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=table_rows[0].keys())
    writer.writeheader()
    writer.writerows(table_rows)
print(f"✓ Table III saved: {csv_path}")

# ── QA panel — GT bubbles coloured by U-Net acceptance ───────────────────────
import random
sample = sorted(random.sample(frame_results, min(12, len(frame_results))),
                key=lambda x: x["file_name"])

n_cols = 4
n_rows = (len(sample) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 3.2, n_rows * 4.5))
axes = np.array(axes).flatten()

for i, r in enumerate(sample):
    ax    = axes[i]
    frame = load_frame(os.path.join(PREPROC_FRAMES, r["file_name"]))
    ax.imshow(frame, cmap="gray", aspect="auto")

    for ann, res in zip(r["gt_anns"], r["ann_results"]):
        bb    = ann["bbox"]
        cx    = bb[0] + bb[2] / 2.0
        cy    = bb[1] + bb[3] / 2.0
        r_bb  = bb[2] / 2.0
        # Green = accepted by U-Net, red = rejected
        color = "lime" if res["accepted"] else "tomato"
        ax.add_patch(Circle((cx, cy), r_bb,
                            fill=False, edgecolor=color, lw=1.2))

    n_acc = r["n_accepted"]
    n_tot = r["n_gt"]
    iou   = r["mean_iou"]
    ax.set_title(
        f"{r['file_name'][-20:]}\n"
        f"acc={n_acc}/{n_tot}  IoU={iou:.3f}",
        fontsize=6.5, pad=2
    )
    ax.axis("off")

for j in range(len(sample), len(axes)):
    axes[j].axis("off")

legend = [
    mpatches.Patch(edgecolor="lime",   facecolor="none",
                   label="U-Net accepted"),
    mpatches.Patch(edgecolor="tomato", facecolor="none",
                   label="U-Net rejected"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=9)
fig.suptitle(
    "NB-Step5 QA — GT bubbles coloured by U-Net acceptance\n"
    "(oracle evaluation: GT centroids fed directly to U-Net)",
    fontsize=9, fontweight="bold"
)
plt.tight_layout(rect=[0, 0.04, 1, 1], pad=0.4)
qa_path = os.path.join(EVAL_DIR, f"qa_eval_{STEP5_TS}.png")
plt.savefig(qa_path, dpi=120, bbox_inches="tight")
plt.show()
plt.close()
print(f"✓ QA panel saved: {qa_path}")

In [ ]:
# ── Cell 9 · Save Evaluation Report & Summary ────────────────────────────────

report = {
    "generated_at":            STEP5_TS,
    "model_path":              MODEL_PATH,
    "coco_gt":                 COCO_PREPROC,
    "n_frames":                len(frame_results),
    "n_gt_total":              results_unet["n_gt_total"],
    "evaluation_mode":         "oracle — GT centroids fed directly to U-Net",
    "results_unet_retrained":  results_unet,
    "results_unet_original": {
        "coverage_pct":    ORIGINAL_UNET_COVERAGE,
        "kl_divergence":   0.697,
        "AP_oracle":       "n/a",
        "mean_patch_iou":  "n/a",
        "note":            "Historical — misaligned training data",
    },
    "per_frame": [
        {
            "image_id":   r["image_id"],
            "file_name":  r["file_name"],
            "n_gt":       r["n_gt"],
            "n_accepted": r["n_accepted"],
            "mean_iou":   r["mean_iou"],
        }
        for r in frame_results
    ],
}

report_path = os.path.join(EVAL_DIR, f"eval_report_{STEP5_TS}.json")
with open(report_path, "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"✓ Evaluation report saved: {report_path}")

W = 70
print()
print("=" * W)
print("  NB-Step5 · EVALUATION SUMMARY".center(W))
print("=" * W)
print(f"\n  {'Method':<38} {'Coverage':>9} {'IoU mean':>9} {'AP oracle':>10}")
print(f"  {'─'*38} {'─'*9} {'─'*9} {'─'*10}")
print(f"  {'U-Net patch (original, misaligned)':<38} "
      f"{ORIGINAL_UNET_COVERAGE:>8.1f}%  "
      f"{'n/a':>9}  "
      f"{'n/a':>10}")
print(f"  {'U-Net patch (retrained, aligned)':<38} "
      f"{results_unet['coverage_pct']:>8.1f}%  "
      f"{results_unet['mean_patch_iou']:>9.4f}  "
      f"{results_unet['AP_oracle']:>10.4f}")

print(f"\n  Coverage gain   : "
      f"+{results_unet['coverage_pct'] - ORIGINAL_UNET_COVERAGE:.1f} pp")
print(f"  IoU >= 0.5      : {results_unet['iou_ge_0p5_pct']:.1f} % of GT bubbles")
print(f"  KL divergence   : {results_unet['kl_divergence']:.3f}")
print(f"  d50             : {results_unet['d50_px']:.2f} px")
print()
print(f"  Table III : {csv_path}")
print(f"  QA panel  : {qa_path}")
print(f"  Report    : {report_path}")
print()
print("  ✓ Step 5 complete.")
print("  Next → update Table III and Results section in the manuscript.")
print("=" * W)

In [ ]:
# ── AP diagnostic — do not skip ───────────────────────────────────────────────

print("=== TP/FP counts ===")
print(f"  CV    n_tp={results_cv['n_tp']}  n_fp={results_cv['n_fp']}  "
      f"n_gt={results_cv['n_gt_total']}")
print(f"  UNet  n_tp={results_unet['n_tp']}  n_fp={results_unet['n_fp']}  "
      f"n_gt={results_unet['n_gt_total']}")

print("\n=== Spatial distribution (y-axis) ===")
gt_ys  = [b[1]+b[3]/2 for r in frame_results for b in r["gt_bboxes"]]
cv_ys  = [d["cy"]     for r in frame_results for d in r["cv_dets"]]
un_ys  = [d["cy"]     for r in frame_results for d in r["unet_dets"]]
print(f"  GT   y — min={min(gt_ys):.0f}  max={max(gt_ys):.0f}  "
      f"mean={np.mean(gt_ys):.0f}  median={np.median(gt_ys):.0f}")
print(f"  CV   y — min={min(cv_ys):.0f}  max={max(cv_ys):.0f}  "
      f"mean={np.mean(cv_ys):.0f}  median={np.median(cv_ys):.0f}")
print(f"  UNet y — min={min(un_ys):.0f}  max={max(un_ys):.0f}  "
      f"mean={np.mean(un_ys):.0f}  median={np.median(un_ys):.0f}")

print("\n=== Closest GT–CV pair across all frames ===")
best_dist = float("inf")
best_info = {}
for r in frame_results:
    for det in r["cv_dets"]:
        for gt_b in r["gt_bboxes"]:
            gt_cx = gt_b[0] + gt_b[2]/2
            gt_cy = gt_b[1] + gt_b[3]/2
            dist  = np.sqrt((det["cx"]-gt_cx)**2 + (det["cy"]-gt_cy)**2)
            r_gt  = 0.5 * np.sqrt(gt_b[2]**2 + gt_b[3]**2)
            if dist < best_dist:
                best_dist = dist
                best_info = {"file":    r["file_name"],
                             "det_cx":  det["cx"],  "det_cy":  det["cy"],
                             "gt_cx":   gt_cx,       "gt_cy":   gt_cy,
                             "dist_px": dist,        "r_gt":    r_gt,
                             "is_tp":   dist <= r_gt}

print(f"  File    : {best_info['file']}")
print(f"  CV det  : ({best_info['det_cx']:.1f}, {best_info['det_cy']:.1f})")
print(f"  GT ctr  : ({best_info['gt_cx']:.1f},  {best_info['gt_cy']:.1f})")
print(f"  Distance: {best_info['dist_px']:.1f} px  |  "
      f"r_gt={best_info['r_gt']:.1f} px  |  TP={best_info['is_tp']}")

In [ ]:
# ── Definitive diagnostic ─────────────────────────────────────────────────────
import random

r = random.choice(frame_results)
frame = load_frame(os.path.join(PREPROC_FRAMES, r["file_name"]))

fig, ax = plt.subplots(1, 1, figsize=(6, 12))
ax.imshow(frame, cmap="gray", aspect="auto")

from matplotlib.patches import Circle
# GT — green
for bb in r["gt_bboxes"]:
    cx, cy = bb[0]+bb[2]/2, bb[1]+bb[3]/2
    ax.add_patch(Circle((cx,cy), bb[2]/2, fill=False, edgecolor="lime", lw=1.5))

# CV — red
for d in r["cv_dets"]:
    ax.add_patch(Circle((d["cx"],d["cy"]), d["diameter"]/2,
                         fill=False, edgecolor="tomato", lw=1.5))

ax.set_title(f"{r['file_name']}\nGT={r['n_gt']} (green)  CV={r['n_cv']} (red)")
ax.axis("off")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/eval/debug_overlay.png",
            dpi=120, bbox_inches="tight")
plt.show(); plt.close()
print("Saved to eval/debug_overlay.png")

In [ ]:
all_gt_y = [b[1] for r in frame_results for b in r["gt_bboxes"]]
all_gt_x = [b[0] for r in frame_results for b in r["gt_bboxes"]]
all_gt_y2 = [b[1]+b[3] for r in frame_results for b in r["gt_bboxes"]]
all_gt_x2 = [b[0]+b[2] for r in frame_results for b in r["gt_bboxes"]]

print(f"GT x range : {min(all_gt_x):.0f} → {max(all_gt_x2):.0f}")
print(f"GT y range : {min(all_gt_y):.0f} → {max(all_gt_y2):.0f}")
print(f"\nSuggested channel ROI (with 20px margin):")
print(f"  x : {max(0, min(all_gt_x)-20):.0f} → "
      f"{min(1080, max(all_gt_x2)+20):.0f}")
print(f"  y : {max(0, min(all_gt_y)-20):.0f} → "
      f"{min(1475, max(all_gt_y2)+20):.0f}")

In [ ]:
import json, os

# Check what inference data is available from the original pipeline
HIDRO_BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026"

# List all relevant directories
for root, dirs, files in os.walk(HIDRO_BASE):
    depth = root.replace(HIDRO_BASE, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth <= 1:
        for f in sorted(files):
            print(f"{indent}  {f}")

In [ ]:
import os

SETUP_A = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A"

for run_dir in sorted(os.listdir(SETUP_A)):
    run_path = os.path.join(SETUP_A, run_dir)
    if not os.path.isdir(run_path):
        continue
    files = sorted(os.listdir(run_path))
    print(f"\n{run_dir}/")
    for f in files:
        fpath = os.path.join(run_path, f)
        size  = os.path.getsize(fpath)
        print(f"  {f:<50} {size/1024:>8.1f} KB")

In [ ]:
import json

NB_PATH = ("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/"
           "04_temporal_model_up6_clean_final.ipynb")

with open(NB_PATH) as f:
    nb = json.load(f)

# Print first non-empty line of each code cell — first 10 cells only
code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]
for i, c in enumerate(code_cells[:10]):
    src = [l for l in c["source"] if l.strip()]
    if src:
        print(f"[{i}] {src[0][:120]}")

In [ ]:
import os, json

BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/outputs"

# Check tracking directories
for d in ["tracking", "tracking_output"]:
    path = os.path.join(BASE, d)
    print(f"\n{d}/")
    if os.path.isdir(path):
        for f in sorted(os.listdir(path)):
            fpath = os.path.join(path, f)
            size  = os.path.getsize(fpath) / 1024
            print(f"  {f:<50} {size:>8.1f} KB")
    else:
        print("  (empty or missing)")

# Read config
cfg_path = os.path.join(BASE, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
print(f"\nconfig.json keys: {list(cfg.keys())}")
print(json.dumps(cfg, indent=2)[:2000])

In [ ]:
import pandas as pd

# Try common track file locations and formats
candidates = [
    f"{BASE}/tracking/tracks.csv",
    f"{BASE}/tracking/tracks.parquet",
    f"{BASE}/tracking_output/tracks.csv",
    f"{BASE}/tracking_output/tracks.parquet",
]

for path in candidates:
    if os.path.exists(path):
        print(f"Found: {path}")
        if path.endswith(".csv"):
            df = pd.read_csv(path, nrows=5)
        else:
            df = pd.read_parquet(path).head(5)
        print(f"  Shape   : {df.shape}")
        print(f"  Columns : {list(df.columns)}")
        print(f"  Dtypes  :\n{df.dtypes}")
        print(f"\n  First 3 rows:")
        print(df.head(3).to_string())
        break
else:
    print("No tracks file found in expected locations.")
    print("Listing all files recursively under outputs/:")
    for root, dirs, files in os.walk(BASE):
        for f in files:
            fp = os.path.join(root, f)
            print(f"  {fp.replace(BASE,'')}  "
                  f"{os.path.getsize(fp)/1024:.1f} KB")

In [ ]:
import json

NB_PATH = ("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/"
           "04_temporal_model_up6_clean_final.ipynb")

with open(NB_PATH) as f:
    nb = json.load(f)

code_cells = [c for c in nb["cells"] if c["cell_type"] == "code"]
for i, c in enumerate(code_cells[9:25]):
    src = [l.rstrip() for l in c["source"] if l.strip()]
    if src:
        print(f"\n[{i+9}]")
        for l in src[:6]:
            print(f"  {l}")

In [ ]:
import os, pandas as pd

BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/campaigns/setup_A/outputs"

# Check temporal directories
for d in ["temporal", "temporal_output"]:
    path = os.path.join(BASE, d)
    print(f"\n{d}/")
    if os.path.isdir(path):
        files = os.listdir(path)
        if files:
            for f in sorted(files):
                fp = os.path.join(path, f)
                print(f"  {f:<50} {os.path.getsize(fp)/1024:>8.1f} KB")
        else:
            print("  (empty)")

# Check if tracks_temporal exists anywhere
print("\nSearching for tracks_temporal.parquet ...")
for root, dirs, files in os.walk(BASE):
    for f in files:
        if "temporal" in f and f.endswith(".parquet"):
            fp = os.path.join(root, f)
            df = pd.read_parquet(fp)
            print(f"  Found: {fp.replace(BASE,'')}")
            print(f"  Shape: {df.shape}")
            print(f"  Cols : {list(df.columns)}")
            break

In [ ]:
import pandas as pd
import numpy as np

TRACKS = ("/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/"
          "campaigns/setup_A/outputs/temporal/tracks_temporal.parquet")

df = pd.read_parquet(TRACKS)

print(f"Total rows          : {len(df)}")
print(f"Unique tracks       : {df['global_bubble_id'].nunique()}")
print(f"Frames              : {df['frame_id'].nunique()}")
print(f"\nae_anomaly_flag counts:")
print(df["ae_anomaly_flag"].value_counts())
print(f"\nae_reconstruction_error stats:")
print(df["ae_reconstruction_error"].describe())
print(f"\nupward_speed_mm_s stats (all tracks):")
print(df["upward_speed_mm_s"].describe())
print(f"\nupward_speed_mm_s stats (normal tracks only):")
print(df[df["ae_anomaly_flag"]==False]["upward_speed_mm_s"].describe())
print(f"\ntrajectory_consistency present: "
      f"{'trajectory_consistency' in df.columns}")
if "trajectory_consistency" in df.columns:
    print(df["trajectory_consistency"].describe())